# RAG agent benchmark · one Golden set, interchangeable agents

Every team provides **one importable function** `answer(question)`. It can
be synchronous or asynchronous and returns either a string or a mapping
with `answer` and optional actual `retrieval_context` passages.

This notebook knows nothing about the agents' classes, frameworks or tools.
It asks each agent the **same native DeepEval Goldens**, checkpoints answers,
and calls `deepeval.evaluate()` separately for factual correctness,
retrieval grounding and optionally style compliance. It does **not** use
`DeepEvalTarget`, `evals_iterator()`, or generator-script imports.

## 1 · Choose the participating agents

To add an agent, install its package in this Jupyter environment and add a
single entry under `AGENTS`. The module must export `answer(question)`.
Use a **distinct agent ID and revision** for reproducible response caching.
A new revision is required after changes to agent code or configuration.

The built-in `cli-agent` entry is a working example; its implementation
is contained entirely in `create_agent()`. Replace or remove that entry
for a team-only benchmark. `MAX_CASES=3` is a smoke test; use `None`
for the full shared Golden set.

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import importlib
import inspect
import json
import os
import re
import tempfile
import tomllib
from dataclasses import asdict, is_dataclass
from collections.abc import Mapping

WORKSPACE = Path.cwd().resolve()   # workspace containing rag-goldens/
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "deepeval-goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

AGENTS = {
    "cli-agent": {
        "entrypoint": "builtin:cli-agent",
        "revision": "sdk-baseline-v1",
        "system_prompt": None,      # optional, only for built-in cli-agent
        "style_criteria": None,     # optional; omit for common-only metrics
    },
    # "team-b": {
    #     "entrypoint": "team_b.benchmark:answer",
    #     "revision": "commit-a1b2c3d",
    #     "style_criteria": None,
    # },
}

# Used only by the optional builtin:cli-agent example
CONFIG_FILE = None
ADMIN_CONFIG_FILE = None
APPROVE_TOOLS = ()

MAX_CASES = 3
TIMEOUT_SECONDS = 600
FRESH_RESPONSES = False
CACHE_EPOCH = ""  # update after changing corpus, indexes or external services

# Same independently chosen judge for every agent
JUDGE_PROVIDER = "openrouter"  # openrouter | openai | ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False

## 2 · Load one immutable DeepEval dataset and an independent judge

No agent code is needed to load the dataset. DeepEval owns only the
**evaluation stage**: the agents never see `expected_output`.

In [ ]:
from deepeval.dataset import EvaluationDataset
from deepeval.models import OpenAIModel, OpenRouterModel, OllamaModel

dataset = EvaluationDataset()
dataset.add_goldens_from_json_file(str(GOLDENS_FILE))
if MAX_CASES is not None:
    dataset = EvaluationDataset(goldens=dataset.goldens[:MAX_CASES])
if not dataset.goldens:
    raise ValueError("The shared Golden set is empty")
if len({g.additional_metadata["golden_id"] for g in dataset.goldens}) != len(dataset.goldens):
    raise ValueError("Duplicate golden_id in the dataset")

if not AGENTS or any(not re.fullmatch(r"[a-z][a-z0-9_-]{0,40}", name) for name in AGENTS):
    raise ValueError("Agent IDs must be unique lowercase identifiers")
for name, spec in AGENTS.items():
    if not isinstance(spec, dict):
        raise TypeError(f"{name}: agent specification must be a dictionary")
    if not isinstance(spec.get("entrypoint"), str) or not spec["entrypoint"].strip():
        raise ValueError(f"{name}: entrypoint is required")
    if not isinstance(spec.get("revision"), str) or not spec["revision"].strip():
        raise ValueError(f"{name}: nonempty revision is required for cache correctness")
    style = spec.get("style_criteria")
    if style is not None and (not isinstance(style, str) or not style.strip()):
        raise ValueError(f"{name}: style_criteria must be text or None")

if JUDGE_PROVIDER == "openrouter":
    if JUDGE_BASE_URL:
        raise ValueError("OpenRouter uses its fixed endpoint")
    judge = OpenRouterModel(model=JUDGE_MODEL, api_key=os.environ["OPENROUTER_API_KEY"])
elif JUDGE_PROVIDER == "openai":
    judge = OpenAIModel(
        model=JUDGE_MODEL, api_key=os.environ["OPENAI_API_KEY"],
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
elif JUDGE_PROVIDER == "ollama":
    judge = OllamaModel(
        model=JUDGE_MODEL,
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
else:
    raise ValueError(f"Unsupported judge provider: {JUDGE_PROVIDER}")
print(f"{len(dataset.goldens)} identical Goldens | {len(AGENTS)} agents | judge={JUDGE_MODEL}")

## 3 · One agent adapter function – the only integration boundary

An external team's installed module provides one `answer(question)`.
For example, `team_b.benchmark:answer` names the function `answer` inside
Python module `team_b.benchmark`. No class, inheritance, DeepEval import,
`sys.path` manipulation or special SDK is required.

`create_agent(spec)` resolves and returns that callable. The optional
`builtin:cli-agent` branch illustrates an implementation using the SDK,
including safe extraction of **actually selected** Knowledge passages.
The benchmark never fabricates retrieval from Golden reference quotations.

Adapters should establish an independent question/session lifecycle so that
questions cannot influence one another through conversation history.

In [ ]:
def cli_agent_retrieval_context(trace):
    """Use only complete, host-verified SDK selected passages."""
    if is_dataclass(trace):
        trace = asdict(trace)
    if not isinstance(trace, dict) or trace.get("truncated") is True:
        return None
    if trace.get("content_included") is not True:
        return None
    selected = trace.get("selected")
    if not isinstance(selected, (list, tuple)) or not selected:
        return None
    passages = [hit.get("content") if isinstance(hit, dict) else None for hit in selected]
    if any(not isinstance(item, str) or not item.strip() for item in passages):
        return None
    return passages


def create_agent(spec):
    """Return any sync or async callable accepting ONLY question: str.

    External agents must be importable Python modules (install their package
    in the notebook environment). Each adapter owns its setup/cleanup and
    must NOT access Golden expected answers.
    """
    entrypoint = spec["entrypoint"]
    if entrypoint == "builtin:cli-agent":
        from cli_agent.sdk import AgentBuilder
        from cli_agent.settings.config import ensure_default_config_file

        prompt = spec.get("system_prompt")
        if prompt is not None and (not isinstance(prompt, str) or not prompt.strip()):
            raise ValueError("builtin cli-agent system_prompt must be text or None")

        async def answer(question):
            project_config = Path(CONFIG_FILE) if CONFIG_FILE else ensure_default_config_file()
            admin_config = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE else None
            builder = AgentBuilder().with_admin_config(path=admin_config)
            builder.with_project_config(
                path=project_config, workspace=WORKSPACE,
                workspace_access="none", response_format="text",
            )
            if APPROVE_TOOLS:
                allowed = frozenset(APPROVE_TOOLS)
                async def approve(name, arguments):
                    return name in allowed
                builder.with_approval_callback(approve)
            if prompt is not None:
                builder.with_system_prompt(prompt)
            agent = builder.with_knowledge_trace(include_content=True).build()
            async with agent:  # fresh independent conversation per question
                result = await agent.ask(question)
            return {
                "answer": result.answer,
                "retrieval_context": cli_agent_retrieval_context(result.knowledge_trace),
            }
        return answer

    if entrypoint.count(":") != 1:
        raise ValueError("entrypoint must be 'python.module:function'")
    module_name, function_name = entrypoint.split(":", 1)
    if not module_name or not all(p.isidentifier() for p in module_name.split(".")):
        raise ValueError(f"Invalid agent module in {entrypoint!r}")
    if not function_name.isidentifier():
        raise ValueError(f"Invalid agent function in {entrypoint!r}")
    function = getattr(importlib.import_module(module_name), function_name)
    if not callable(function):
        raise TypeError(f"{entrypoint} is not callable")
    return function


def normalize_answer(value):
    """Accepted outputs: str or {'answer': str, 'retrieval_context': list[str]}."""
    if isinstance(value, str):
        answer, context = value, None
    elif isinstance(value, Mapping):
        answer = value.get("answer")
        context = value.get("retrieval_context")
    else:
        raise TypeError("Agent must return a string or an answer mapping")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("Agent returned an empty answer")
    if context is not None:
        if (not isinstance(context, (list, tuple))
                or not context or any(
                    not isinstance(part, str) or not part.strip()
                    for part in context
                )):
            raise ValueError("retrieval_context must contain nonempty strings")
        context = list(context)
    return {"actual_output": answer, "retrieval_context": context}


async def invoke_agent(answer_fn, question):
    """Support sync + async functions; time out individual cases."""
    if inspect.iscoroutinefunction(answer_fn):
        call = answer_fn(question)
    else:
        call = asyncio.to_thread(answer_fn, question)
    result = await asyncio.wait_for(call, timeout=TIMEOUT_SECONDS)
    if inspect.isawaitable(result):
        result = await asyncio.wait_for(result, timeout=TIMEOUT_SECONDS)
    return normalize_answer(result)

## 4 · Execute all agents against the same Goldens

Target execution is independent of DeepEval: it calls only
`answer(question)`, optionally records actual retrieval passages,
and saves a private, atomic response checkpoint for each agent/question.

The cache key includes `entrypoint`, `revision`, agent ID, Golden ID,
workspace and `CACHE_EPOCH`. For the built-in SDK adapter it also hashes
effective config/admin/prompt files. External teams must update their
`revision` after code/config changes (or set `FRESH_RESPONSES=True`).
Failures are counted and displayed rather than silently excluded.

A sync adapter can block in a worker thread after its timeout; teams should
provide cooperative cancellation for long-running processes.

In [ ]:
def agent_fingerprint(name, spec):
    digest = hashlib.sha256(json.dumps({
        "name": name, "entrypoint": spec["entrypoint"],
        "revision": spec["revision"], "system_prompt": spec.get("system_prompt"),
        "workspace": str(WORKSPACE), "epoch": CACHE_EPOCH,
    }, sort_keys=True).encode("utf-8"))
    if spec["entrypoint"] == "builtin:cli-agent":
        # Other teams' adapters do not need cli-agent installed at all.
        from cli_agent.settings.admin_config import default_admin_config_file
        from cli_agent.settings.config import ensure_default_config_file
        config = Path(CONFIG_FILE) if CONFIG_FILE else ensure_default_config_file()
        admin = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE else default_admin_config_file()
        files = [config]
        if admin.is_file():
            files.append(admin)
        if spec.get("system_prompt") is None:
            values = tomllib.loads(Path(config).read_text(encoding="utf-8"))
            prompt_file = values.get("agent", {}).get("system_prompt")
            if prompt_file:
                files.append(WORKSPACE / ".cli-agent" / prompt_file)
        for file in files:
            file = Path(file).expanduser().resolve(strict=True)
            digest.update(str(file).encode())
            digest.update(file.read_bytes())
        digest.update(json.dumps(sorted(APPROVE_TOOLS)).encode())
    return digest.hexdigest()


def response_file(agent_id, spec, golden):
    key = hashlib.sha256(json.dumps([
        agent_fingerprint(agent_id, spec),
        golden.additional_metadata["golden_id"],
        golden.input, golden.expected_output,
    ], ensure_ascii=False).encode("utf-8")).hexdigest()
    return OUTPUT_DIR / "responses" / f"{key}.json"


def read_cached_response(path):
    if not path.is_file() or path.is_symlink() or path.stat().st_size > 16_000_000:
        return None
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(record, dict):
            return normalize_answer({
                "answer": record.get("actual_output"),
                "retrieval_context": record.get("retrieval_context"),
            })
    except (ValueError, TypeError, OSError):
        pass
    return None


def save_response(path, response):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=path.parent,
            prefix=".response-", suffix=".tmp", delete=False,
        ) as stream:
            temporary = Path(stream.name)
            os.chmod(temporary, 0o600)
            json.dump(response, stream, ensure_ascii=False)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


all_responses = {}
all_failures = {}
for agent_id, spec in AGENTS.items():
    answer_fn = create_agent(spec)  # agent-specific details end HERE
    responses, failures = {}, {}
    for golden in dataset.goldens:
        golden_id = golden.additional_metadata["golden_id"]
        checkpoint = response_file(agent_id, spec, golden)
        try:
            response = None if FRESH_RESPONSES else read_cached_response(checkpoint)
            if response is None:
                response = await invoke_agent(answer_fn, golden.input)
                save_response(checkpoint, response)
            responses[golden_id] = response
        except Exception as error:
            # Do not log potentially sensitive question/answer/trace content.
            failures[golden_id] = type(error).__name__
    all_responses[agent_id] = responses
    all_failures[agent_id] = failures
    print(f"{agent_id}: {len(responses)}/{len(dataset.goldens)} answered; "
          f"{len(failures)} failed")
    if failures:
        print("  Failed Golden IDs:", ", ".join(sorted(failures)))

## 5 · DeepEval exclusively evaluates saved agent responses

Every agent is judged against the **same unchanged reference answers** using
the **same judge model and common rubrics**. Retrieval metrics use only
passages that the agent actually supplied. Style compliance is optional
and agent-specific; do not rank different style rubrics as if they were one
common metric.

DeepEval handles metric execution, caching and reports. No observed
agent tracing is required; the answer and actual retrieval passages are
provided directly through `LLMTestCase`.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    GEval, AnswerRelevancyMetric, FaithfulnessMetric,
    ContextualRecallMetric, ContextualPrecisionMetric,
)
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

evaluation_options = {
    "cache_config": CacheConfig(use_cache=not FRESH_EVALUATION, write_cache=True),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"),
        inspect_after_run=False,
    ),
}

evaluation_results = {}
for agent_id, spec in AGENTS.items():
    responses = all_responses[agent_id]
    answer_cases, retrieval_cases = [], []
    for golden in dataset.goldens:
        response = responses.get(golden.additional_metadata["golden_id"])
        if response is None:
            continue  # failures are counted separately in the summary
        base = {
            "input": golden.input,
            "expected_output": golden.expected_output,
            "actual_output": response["actual_output"],
        }
        answer_cases.append(LLMTestCase(**base))
        if response["retrieval_context"]:
            retrieval_cases.append(LLMTestCase(
                **base, retrieval_context=response["retrieval_context"],
            ))

    correctness = GEval(
        name="Answer correctness",
        criteria=(
            "Judge factual correctness and completeness against the fixed reference. "
            "Accept paraphrases, short or detailed answers equally when all "
            "necessary facts and caveats are preserved. Penalize invented facts."
        ),
        evaluation_params=[
            SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            SingleTurnParams.EXPECTED_OUTPUT,
        ],
        model=judge, threshold=THRESHOLD,
    )
    metrics = [
        correctness,
        AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
    ]
    if spec.get("style_criteria"):
        metrics.append(GEval(
            name="Style compliance",
            criteria=(
                "Score only response style, not factual correctness. Goal: "
                + spec["style_criteria"]
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            ],
            model=judge, threshold=THRESHOLD,
        ))

    answer_results = evaluate(
        test_cases=answer_cases, metrics=metrics,
        identifier=f"rag-{agent_id}-answer",
        async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
        **evaluation_options,
    ) if answer_cases else None

    retrieval_results = None
    if retrieval_cases:
        uncertainty = GEval(
            name="Uncertainty calibration",
            criteria=(
                "Using the actual retrieved passages, assess justified "
                "confidence or limitations in the answer. Penalize unsupported "
                "claims and unnecessary refusal when retrieved evidence supports "
                "an answer. The reference is the factual target, NOT proof that "
                "the agent retrieved it."
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT,
            ],
            model=judge, threshold=THRESHOLD,
        )
        retrieval_results = evaluate(
            test_cases=retrieval_cases,
            metrics=[
                FaithfulnessMetric(model=judge, threshold=THRESHOLD),
                ContextualRecallMetric(model=judge, threshold=THRESHOLD),
                ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
                uncertainty,
            ],
            identifier=f"rag-{agent_id}-retrieval",
            async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
            **evaluation_options,
        )
    evaluation_results[agent_id] = {
        "answer": answer_results,
        "retrieval": retrieval_results,
        "successful_cases": len(answer_cases),
        "retrieval_cases": len(retrieval_cases),
    }

## 6 · Compare results and coverage

Metrics are averages across successfully judged answers. The table **always
shows missing/failed agent questions** so that an agent that answered fewer
Goldens cannot appear to win by only answering easy questions. Averages
based on different answered subsets are not directly comparable until
coverage is equal. Retrieval evaluation counts are shown separately.

In [ ]:
from collections import defaultdict
import pandas as pd
from IPython.display import display

summary_rows = []
for agent_id, results in evaluation_results.items():
    scores = defaultdict(list)
    judge_errors = defaultdict(int)
    for section in ("answer", "retrieval"):
        run = results[section]
        for testcase in (getattr(run, "test_results", []) if run else []):
            for metric in testcase.metrics_data or []:
                name = metric.name.removesuffix(" [GEval]")
                if metric.error or metric.score is None:
                    judge_errors[name] += 1
                else:
                    scores[name].append(float(metric.score))
    row = {
        "agent": agent_id,
        "goldens": len(dataset.goldens),
        "answered": results["successful_cases"],
        "agent_failures": len(all_failures[agent_id]),
        "retrieval_cases": results["retrieval_cases"],
    }
    for name in (
        "Answer correctness", "Answer Relevancy",
        "Faithfulness", "Contextual Recall", "Contextual Precision",
        "Uncertainty calibration", "Style compliance",
    ):
        values = scores.get(name, [])
        row[name] = round(sum(values) / len(values), 3) if values else None
        row[name + " judged"] = len(values)
        row[name + " judge_errors"] = judge_errors[name]
    summary_rows.append(row)

display(pd.DataFrame(summary_rows))
print("Reports:", OUTPUT_DIR / "deepeval")

### Interpretation and reproducibility

- Everyone uses the same Golden questions, unchanged references, judge
  model and shared factual rubrics. Each target's setup remains its team's
  responsibility.
- For controlled comparisons, use a fixed corpus/index snapshot, model
  settings and adapter `revision`, and set `MAX_CASES=None`.
- A result without `retrieval_context` can still be judged for correctness,
  but not for actual retrieval grounding. Never substitute Golden quotations.
- Changing another team's implementation, prompt, model or external index
  requires an updated `revision` / `CACHE_EPOCH` or
  `FRESH_RESPONSES=True`. Judge cache is independent.
- Stored responses and DeepEval reports may contain sensitive data. Keep
  `rag-goldens/evaluation/` out of version control.